In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

data = pd.read_csv('data-1.csv')

print(f"Original total samples: {len(data)}")
print(f"Label distribution:\n{data['label'].value_counts()}")

compound_column = 'name'
X = data.drop(['label'], axis=1)
y = data['label']
compounds = X[compound_column]

unique_compounds = compounds.unique()
print(f"\nTotal unique compounds: {len(unique_compounds)}")

compound_to_label = {}
for compound in unique_compounds:
    compound_mask = compounds == compound
    compound_labels = y[compound_mask]
    majority_label = compound_labels.mode()[0] if not compound_labels.mode().empty else compound_labels.iloc[0]
    compound_to_label[compound] = majority_label

compound_labels = [compound_to_label[c] for c in unique_compounds]

unique_compounds_train, unique_compounds_test, _, _ = train_test_split(
    unique_compounds,
    compound_labels,
    test_size=0.2,
    random_state=42,
    stratify=compound_labels
)

train_mask = compounds.isin(unique_compounds_train)
test_mask = compounds.isin(unique_compounds_test)

X_train_raw = X[train_mask].copy()
X_test_raw = X[test_mask].copy()
y_train = y[train_mask].copy()
y_test = y[test_mask].copy()

print(f"\nTraining set samples: {len(X_train_raw)}")
print(f"Test set samples: {len(X_test_raw)}")
print(f"Training set unique compounds: {len(unique_compounds_train)}")
print(f"Test set unique compounds: {len(unique_compounds_test)}")
print(f"Training set label distribution: {y_train.value_counts().to_dict()}")
print(f"Test set label distribution: {y_test.value_counts().to_dict()}")

print("\n" + "="*50)
print("Training set compound examples (first 10):")
print("="*50)
for i, name in enumerate(X_train_raw[compound_column].head(10), 1):
    print(f"{i}. {name}")

print("\n" + "="*50)
print("Test set compound examples (first 10):")
print("="*50)
for i, name in enumerate(X_test_raw[compound_column].head(10), 1):
    print(f"{i}. {name}")

train_names_saved = X_train_raw[compound_column].copy()
test_names_saved = X_test_raw[compound_column].copy()

X_train = X_train_raw.drop(columns=[compound_column])
X_test = X_test_raw.drop(columns=[compound_column])

for column in X_train.columns:
    if X_train[column].dtype in ['float64', 'int64']:
        if X_train[column].notna().any():
            median_value = X_train[column].median()
        else:
            median_value = 0
        X_train[column].fillna(median_value, inplace=True)

for column in X_test.columns:
    if X_test[column].dtype in ['float64', 'int64'] and column in X_train.columns:
        if X_train[column].notna().any():
            median_value = X_train[column].median()
        else:
            median_value = 0
        X_test[column].fillna(median_value, inplace=True)

continuous_features_train = X_train.select_dtypes(include=['float64', 'int64']).columns
continuous_features_test = X_test.select_dtypes(include=['float64', 'int64']).columns
discrete_features_train = X_train.select_dtypes(include=['object']).columns
discrete_features_test = X_test.select_dtypes(include=['object']).columns

print(f"\nContinuous features: {len(continuous_features_train)}")
print(f"Discrete features: {len(discrete_features_train)}")
if len(discrete_features_train) > 0:
    print(f"Discrete feature examples: {list(discrete_features_train)[:5]}")

scaler = StandardScaler()
X_train_continuous = scaler.fit_transform(X_train[continuous_features_train])
X_test_continuous = scaler.transform(X_test[continuous_features_test])

X_train_processed = pd.DataFrame(
    X_train_continuous, 
    columns=continuous_features_train, 
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_continuous, 
    columns=continuous_features_test, 
    index=X_test.index
)

if len(discrete_features_train) > 0:
    print(f"\nDiscrete features detected: {list(discrete_features_train)}")
    
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    train_discrete_encoded = encoder.fit_transform(X_train[discrete_features_train])
    test_discrete_encoded = encoder.transform(X_test[discrete_features_test])
    
    discrete_columns = []
    for i, col in enumerate(discrete_features_train):
        categories = encoder.categories_[i]
        for cat in categories:
            discrete_columns.append(f"{col}_{cat}")
    
    train_discrete_df = pd.DataFrame(
        train_discrete_encoded,
        columns=discrete_columns,
        index=X_train.index
    )
    test_discrete_df = pd.DataFrame(
        test_discrete_encoded,
        columns=discrete_columns,
        index=X_test.index
    )
    
    X_train_processed = pd.concat([X_train_processed, train_discrete_df], axis=1)
    X_test_processed = pd.concat([X_test_processed, test_discrete_df], axis=1)

X_train = X_train_processed
X_test = X_test_processed

print(f"\nProcessed training set shape: {X_train.shape}")
print(f"Processed test set shape: {X_test.shape}")

print("\n" + "="*50)
print("Final Data Statistics (Compound-Aware Split)")
print("="*50)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training set features: {X_train.shape[1]}")
print(f"Test set features: {X_test.shape[1]}")
print("\nTraining set label distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest set label distribution:")
print(y_test.value_counts(normalize=True))

train_info = pd.DataFrame({
    'name': train_names_saved.values,
    'original_index': train_names_saved.index,
    'label': y_train.values
})

test_info = pd.DataFrame({
    'name': test_names_saved.values,
    'original_index': test_names_saved.index,
    'label': y_test.values
})

train_unique = train_info.drop_duplicates(subset=['name'])
test_unique = test_info.drop_duplicates(subset=['name'])

print("\n" + "="*50)
print(f"Training set unique compounds: {len(train_unique)}")
print(f"Test set unique compounds: {len(test_unique)}")
print("\nTraining set compound examples (first 10):")
print("="*50)
print(train_unique[['name', 'label']].head(10).to_string(index=False))

print("\n" + "="*50)
print("Test set compound examples (first 10):")
print("="*50)
print(test_unique[['name', 'label']].head(10).to_string(index=False))

train_info.to_csv('train_names_no_smote.csv', index=False)
test_info.to_csv('test_names_no_smote.csv', index=False)
train_unique.to_csv('train_unique_names_no_smote.csv', index=False)
test_unique.to_csv('test_unique_names_no_smote.csv', index=False)

print("\n✓ Saved files:")
print("  - train_names_no_smote.csv")
print("  - test_names_no_smote.csv")
print("  - train_unique_names_no_smote.csv")
print("  - test_unique_names_no_smote.csv")

train_compounds_set = set(train_unique['name'])
test_compounds_set = set(test_unique['name'])
overlap_compounds = train_compounds_set & test_compounds_set

print(f"\nOverlapping compounds between train and test: {len(overlap_compounds)}")
if len(overlap_compounds) == 0:
    print("✓ No data leakage - compounds are completely separated!")
else:
    print(f"⚠ Warning: {len(overlap_compounds)} compounds appear in both sets")

print("\n" + "="*50)
print("Code completed successfully!")
print("="*50)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

data = pd.read_csv('data-1.csv')

print(f"Original total samples: {len(data)}")
print(f"Label distribution:\n{data['label'].value_counts()}")

compound_column = 'name'
X = data.drop(['label'], axis=1)
y = data['label']
compounds = X[compound_column]

unique_compounds = compounds.unique()
print(f"\nTotal unique compounds: {len(unique_compounds)}")

compound_to_label = {}
for compound in unique_compounds:
    compound_mask = compounds == compound
    compound_labels = y[compound_mask]
    majority_label = compound_labels.mode()[0] if not compound_labels.mode().empty else compound_labels.iloc[0]
    compound_to_label[compound] = majority_label

compound_labels = [compound_to_label[c] for c in unique_compounds]

unique_compounds_train, unique_compounds_test, _, _ = train_test_split(
    unique_compounds,
    compound_labels,
    test_size=0.2,
    random_state=42,
    stratify=compound_labels
)

train_mask = compounds.isin(unique_compounds_train)
test_mask = compounds.isin(unique_compounds_test)

X_train_raw = X[train_mask].copy()
X_test_raw = X[test_mask].copy()
y_train = y[train_mask].copy()
y_test = y[test_mask].copy()

print(f"\nTraining set samples: {len(X_train_raw)}")
print(f"Test set samples: {len(X_test_raw)}")
print(f"Training set unique compounds: {len(unique_compounds_train)}")
print(f"Test set unique compounds: {len(unique_compounds_test)}")
print(f"Training set label distribution: {y_train.value_counts().to_dict()}")
print(f"Test set label distribution: {y_test.value_counts().to_dict()}")

print("\n" + "="*50)
print("Training set compound examples (first 10):")
print("="*50)
for i, name in enumerate(X_train_raw[compound_column].head(10), 1):
    print(f"{i}. {name}")

print("\n" + "="*50)
print("Test set compound examples (first 10):")
print("="*50)
for i, name in enumerate(X_test_raw[compound_column].head(10), 1):
    print(f"{i}. {name}")

train_names_saved = X_train_raw[compound_column].copy()
test_names_saved = X_test_raw[compound_column].copy()

X_train = X_train_raw.drop(columns=[compound_column])
X_test = X_test_raw.drop(columns=[compound_column])

for column in X_train.columns:
    if X_train[column].dtype in ['float64', 'int64']:
        if X_train[column].notna().any():
            median_value = X_train[column].median()
        else:
            median_value = 0
        X_train[column].fillna(median_value, inplace=True)

for column in X_test.columns:
    if X_test[column].dtype in ['float64', 'int64'] and column in X_train.columns:
        if X_train[column].notna().any():
            median_value = X_train[column].median()
        else:
            median_value = 0
        X_test[column].fillna(median_value, inplace=True)

continuous_features_train = X_train.select_dtypes(include=['float64', 'int64']).columns
continuous_features_test = X_test.select_dtypes(include=['float64', 'int64']).columns
discrete_features_train = X_train.select_dtypes(include=['object']).columns
discrete_features_test = X_test.select_dtypes(include=['object']).columns

print(f"\nContinuous features: {len(continuous_features_train)}")
print(f"Discrete features: {len(discrete_features_train)}")
if len(discrete_features_train) > 0:
    print(f"Discrete feature examples: {list(discrete_features_train)[:5]}")

scaler = StandardScaler()
X_train_continuous = scaler.fit_transform(X_train[continuous_features_train])
X_test_continuous = scaler.transform(X_test[continuous_features_test])

X_train_processed = pd.DataFrame(
    X_train_continuous, 
    columns=continuous_features_train, 
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_continuous, 
    columns=continuous_features_test, 
    index=X_test.index
)

if len(discrete_features_train) > 0:
    print(f"\nDiscrete features detected: {list(discrete_features_train)}")
    
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    train_discrete_encoded = encoder.fit_transform(X_train[discrete_features_train])
    test_discrete_encoded = encoder.transform(X_test[discrete_features_test])
    
    discrete_columns = []
    for i, col in enumerate(discrete_features_train):
        categories = encoder.categories_[i]
        for cat in categories:
            discrete_columns.append(f"{col}_{cat}")
    
    train_discrete_df = pd.DataFrame(
        train_discrete_encoded,
        columns=discrete_columns,
        index=X_train.index
    )
    test_discrete_df = pd.DataFrame(
        test_discrete_encoded,
        columns=discrete_columns,
        index=X_test.index
    )
    
    X_train_processed = pd.concat([X_train_processed, train_discrete_df], axis=1)
    X_test_processed = pd.concat([X_test_processed, test_discrete_df], axis=1)

X_train = X_train_processed
X_test = X_test_processed

print(f"\nProcessed training set shape: {X_train.shape}")
print(f"Processed test set shape: {X_test.shape}")

print("\n" + "="*50)
print("Starting undersampling...")
print("="*50)

print(f"Positive samples before undersampling: {(y_train == 1).sum()}")
print(f"Negative samples before undersampling: {(y_train == 0).sum()}")

undersampler = RandomUnderSampler(random_state=42)
X_train, y_train = undersampler.fit_resample(X_train, y_train)

print(f"\nTraining set shape after undersampling: {X_train.shape}")
print("Label distribution after undersampling:")
print(pd.Series(y_train).value_counts())

print("\n" + "="*50)
print("Final Data Statistics (Undersampled)")
print("="*50)
print(f"Final training set shape: {X_train.shape}")
print(f"Final test set shape: {X_test.shape}")
print(f"Training set features: {X_train.shape[1]}")
print(f"Test set features: {X_test.shape[1]}")
print("\nTraining set label distribution (balanced after undersampling):")
print(y_train.value_counts(normalize=True))
print("\nTest set label distribution (original imbalanced):")
print(y_test.value_counts(normalize=True))

train_labels_downsampled = pd.DataFrame({
    'label': y_train.values
})

train_labels_downsampled.to_csv('train_labels_downsampled.csv', index=False)

test_info = pd.DataFrame({
    'name': test_names_saved.values,
    'original_index': test_names_saved.index,
    'label': y_test.values
})

test_unique = test_info.drop_duplicates(subset=['name'])

print("\n" + "="*50)
print(f"Test set unique compounds: {len(test_unique)}")
print("\nTest set compound examples (first 10):")
print("="*50)
print(test_unique[['name', 'label']].head(10).to_string(index=False))

test_info.to_csv('test_names_downsampled.csv', index=False)
test_unique.to_csv('test_unique_names_downsampled.csv', index=False)

print("\n✓ Saved files:")
print("  - train_labels_downsampled.csv")
print("  - test_names_downsampled.csv")
print("  - test_unique_names_downsampled.csv")

train_compounds_set = set(train_names_saved[train_mask].values)
test_compounds_set = set(test_names_saved.values)
overlap_compounds = train_compounds_set & test_compounds_set

print(f"\nOverlapping compounds between train and test: {len(overlap_compounds)}")
if len(overlap_compounds) == 0:
    print("✓ No data leakage - compounds are completely separated!")
else:
    print(f"⚠ Warning: {len(overlap_compounds)} compounds appear in both sets")

print(f"\nTest set unique compounds: {len(test_unique)}")

print("\n" + "="*50)
print("Code completed successfully!")
print("="*50)